# Trustworthy CXR VLM — second base model: MAIRA-2 (DR-035)

Runs MAIRA-2 (microsoft/maira-2, chest-specific; IU X-Ray held out by its authors) on the same patients,
inputs (frontal image + indication) and scoring as LLaVA-Med S0, with 5 samples per patient for the S3
uncertainty layer. Outputs go to `results_maira2/` and `phase_maira2_results.zip`.

**Before running:** Accelerator **GPU T4 x2**, Internet **On**, add the IU X-Ray dataset as an input, and add a
Kaggle secret **HF_TOKEN** (Add-ons → Secrets) holding a Hugging Face *read* token of an account that accepted
the MAIRA-2 licence. If the log says **GENERATION NOT FINISHED**, attach this version's output and run again.

In [ ]:
import glob, os, time
START = time.time()
GEN_BUDGET_H = 10.5
REPO_URL = "https://github.com/justinthomas11/mts-medical-vlm.git"
REPO_COMMIT = None
IU_DATASET_DIR = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
WORK = "/kaggle/working/mts-medical-vlm"
RESULTS = "results_maira2"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("IU_DATASET_DIR =", IU_DATASET_DIR, "| HF token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
import os, subprocess, sys
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
if REPO_COMMIT:
    subprocess.run(["git", "-C", WORK, "checkout", REPO_COMMIT], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# MAIRA-2's remote code needs transformers >=4.48,<4.52 (model card); LLaVA-Med is not loaded here.
# Scoring dependencies are installed up front so a broken install fails now, not after generation.
!pip install -q "transformers==4.51.3" accelerate einops "torchxrayvision==1.5.4" "pycocoevalcap==1.2"
!pip install -q --no-deps "f1chexbert==0.0.2" "appdirs==1.4.4"
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# RadGraph's bundled AllenNLP code needs transformers < 5, so it gets its own venv (DR-022).
RG_VENV = "/tmp/radgraph-venv"
RG_PY = os.path.join(RG_VENV, "bin", "python")
if not os.path.exists(RG_PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", "-q", RG_VENV], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "torch==2.14.0",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "-r", "requirements-radgraph.txt"], check=True)
subprocess.run([RG_PY, "-c", "import radgraph, transformers; print('radgraph venv ok, transformers', transformers.__version__)"],
               check=True)
os.environ["RADGRAPH_PYTHON"] = RG_PY

In [ ]:
# Link the raw IU X-Ray files where configs/data_config.yaml expects them (data/raw is git-ignored).
os.makedirs("data/raw/images", exist_ok=True)
for src, dst in [("indiana_reports.csv", "data/raw/indiana_reports.csv"),
                 ("indiana_projections.csv", "data/raw/indiana_projections.csv"),
                 ("images/images_normalized", "data/raw/images/images_normalized")]:
    if not os.path.exists(dst):
        os.symlink(os.path.join(IU_DATASET_DIR, src), dst)
print(len(os.listdir("data/raw/images/images_normalized")), "images linked (expected 7,470)")

In [ ]:
# Rebuild data/processed with the fixed seed, then STOP unless the split is identical to the committed one.
subprocess.run([sys.executable, "src/data/split_dataset.py"], check=True)
check = subprocess.run([sys.executable, "src/pipeline/verify_splits.py"])
assert check.returncode == 0, "Split differs from the committed benchmark — do not run any stage."

In [ ]:
# Full test suite on Kaggle (torch is blocked on the dev laptop). Informational: does not stop the run.
tests = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "-p", "no:cacheprovider"],
                       capture_output=True, text=True)
print(tests.stdout[-3000:])

In [ ]:
import json, shutil
os.environ["MTS_RESULTS_DIR"] = RESULTS
for d in ["s0", "s1", "s3"]:
    os.makedirs(f"{RESULTS}/{d}", exist_ok=True)
for f in ["retrieval_val.csv", "retrieval_test.csv", "retrieval_tuning.json"]:   # for the review-flag features
    shutil.copy(f"results/s1/{f}", f"{RESULTS}/s1/{f}")
best = {}
for f in glob.glob(f"/kaggle/input/**/{RESULTS}/s0/generations_*.jsonl", recursive=True):
    if os.path.basename(f) not in ("generations_test.jsonl", "generations_val.jsonl"):
        continue
    n = sum(1 for line in open(f, encoding="utf-8") if line.strip())
    if n > best.get(os.path.basename(f), ("", -1))[1]:
        best[os.path.basename(f)] = (f, n)
for name, (f, n) in best.items():
    shutil.copy(f, f"{RESULTS}/s0/{name}")
    print(f"resumed {RESULTS}/s0/{name}: {n} patients")

In [ ]:
# Fail fast: 3 val patients with 5 samples, printed in full, before the long run.
r = subprocess.run([sys.executable, "src/pipeline/run_generation.py", "--model", "maira2", "--stage", "s0",
                    "--split", "val", "--quant", "fp16", "--samples", "5", "--limit", "3", "--tag", "smoke3"])
assert r.returncode == 0, "MAIRA-2 smoke test failed: see the traceback above"
for line in open(f"{RESULTS}/s0/generations_val_smoke3.jsonl", encoding="utf-8"):
    g = json.loads(line)
    print(g["uid"], g["n_tokens"], round(g["mean_token_entropy"], 3), g["latency_s"], "|", g["generated_report"][:300])

In [ ]:
DEADLINE = START + GEN_BUDGET_H * 3600

def run(*args):
    cmd = [sys.executable, "src/pipeline/run_generation.py", "--model", "maira2", "--stage", "s0", *args,
           "--quant", "fp16", "--samples", "5", "--deadline", str(DEADLINE)]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

run("--split", "val")
run("--split", "test")

def n_done(p):
    return sum(1 for line in open(p, encoding="utf-8") if line.strip()) if os.path.exists(p) else 0

EXPECTED = {f"{RESULTS}/s0/generations_val.jsonl": 366, f"{RESULTS}/s0/generations_test.jsonl": 734}
COMPLETE = all(n_done(f) == n for f, n in EXPECTED.items())
for f, n in EXPECTED.items():
    print(f"{f}: {n_done(f)}/{n}")
print("GENERATION COMPLETE: scoring follows." if COMPLETE else
      "GENERATION NOT FINISHED: scoring skipped. Attach this version's output as an input and Save & Run All again.")

In [ ]:
FAILED = []
if COMPLETE:
    def py(script, *args):
        cmd = [sys.executable, script, *args]
        print(" ".join(cmd), flush=True)
        if subprocess.run(cmd).returncode != 0:
            FAILED.append(" ".join(args) or script)

    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s0/generations_test.jsonl")
    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s0/generations_val.jsonl", "--uncertainty", "--fit-flagger")
    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s0/generations_test.jsonl", "--uncertainty")
    py("src/pipeline/tune_review_flag.py", "--stage", "s0", "--apply-test")
print("scoring failures:", FAILED or "none")

In [ ]:
import zipfile
out = "/kaggle/working/phase_maira2_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{RESULTS}/**/*", recursive=True)):
        if os.path.isfile(f):
            z.write(f)
    z.writestr(f"{RESULTS}/KAGGLE_RUN.txt",
               subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(out, "| complete:", COMPLETE)